# Week 3 · Tutorial 2 — Density estimates, scatter and line plots, boxplots

**Covers videos 3.3 – 3.5:** kernel density estimation · scatter plots and line plots · boxplots.

**Dataset:** `weather_hourly.csv` — hourly data for a Nordic region, 2022 – 2023. **One row is one hour.**

| column | meaning |
|---|---|
| `temperature_c`, `wind_ms` | weather |
| `irradiance_wm2` | solar irradiance |
| `precip_mm` | precipitation in the hour (about 80 % of hours are exactly 0) |
| `load_mw` | regional electricity load |
| `price_eur_mwh` | day-ahead spot price (can be negative, has rare spikes) |

We draw all figures from the **training part** (first 70 % of the hours). Each ✏️ Your turn asks for a change in a figure *and* a number read from the result.

### How to work with this notebook (no programming experience needed)

* A notebook is a list of **cells**. Click a code cell and press **Shift + Enter** to run it; the result appears below it. Run the cells **from top to bottom**.
* Cells that start with `import` or read a file (`pd.read_csv(...)`) just load tools and data. You do not need to understand every line.
* In every **✏️ Your turn** block you make one small, exactly described change (a number, a word, or a line to paste in), then run the cell again.
  The box **What to change** shows the exact text to type. Lines marked `# TODO` are the ones to edit.
* Write down the numbers that appear — the quiz asks for them. If you break something, re-run the cells above it, or use *Kernel → Restart & Run All*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
from scipy.stats import gaussian_kde

wx = pd.read_csv("data/weather_hourly.csv", parse_dates=["timestamp"]).sort_values("timestamp")
train = wx.iloc[:int(0.7 * len(wx))].copy()
print("training hours:", len(train), "|", train["timestamp"].min().date(), "to", train["timestamp"].max().date())
train.head()

## 1. Kernel density estimation (KDE)

A KDE puts a small smooth "bump" on every observation and adds them up. The **bandwidth** (width of the bumps) is the
one setting that matters: too small → noise, too large → one featureless hill. Here the bandwidth is given as a factor
multiplied by the standard deviation of the data (`bw_method=`).

### ✏️ Your turn 1

Estimate the density of the **wind speed** and count its peaks.

Run the cell with `bw = 0.3`, then with `bw = 0.02` (too small) and with `bw = 1.0` (too large). For each, the cell prints the
**number of local maxima** of the curve and the wind speed (1 decimal) at the highest peak.

In [ ]:
bw = 0.3       # TODO: 0.02, then 1.0

w = train["wind_ms"].values
grid = np.linspace(0, 20, 801)
dens = gaussian_kde(w, bw_method=bw)(grid)

n_peaks = int(((dens[1:-1] > dens[:-2]) & (dens[1:-1] > dens[2:])).sum())
print("bandwidth factor:", bw)
print("number of local maxima:", n_peaks)
print("highest peak at wind speed:", round(grid[dens.argmax()], 1), "m/s")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(grid, dens); ax.hist(w, bins=60, density=True, alpha=0.3)
ax.set_xlabel("wind speed (m/s)"); ax.set_ylabel("density")
plt.show()

### Where the KDE lies to you: a variable with a point mass

Precipitation is bounded below by 0 and about 80 % of the hours are *exactly* 0. A KDE has no idea about this boundary — it
leaks probability into negative precipitation.

### ✏️ Your turn 2

1. Fill in the `TODO` lines: `share_zero` = share of hours with `precip_mm == 0` (3 decimals) and `mass_below_zero` = the
   probability mass the KDE assigns to **negative** values: `kde.integrate_box_1d(-np.inf, 0)` (3 decimals).
2. Run with `bw = 0.3`, then change `bw` to `0.6` and run again.

**What to change** — the two `TODO` lines become:

```python
share_zero = (train["precip_mm"] == 0).mean()
mass_below_zero = kde.integrate_box_1d(-np.inf, 0)
```

In [ ]:
bw = 0.3       # TODO: also 0.6

p = train["precip_mm"].values
kde = gaussian_kde(p, bw_method=bw)

share_zero = np.nan          # TODO
mass_below_zero = np.nan     # TODO

print("bandwidth factor:", bw)
print("share of hours with exactly 0 mm:", round(share_zero, 3))
print("KDE mass below zero             :", round(mass_below_zero, 3))

grid = np.linspace(-3, 8, 400)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(grid, kde(grid)); ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("precipitation (mm/h)")
plt.show()

## 2. Scatter plots and line plots

### Prices usually want a log axis

Spot prices are right-skewed with rare spikes: most hours lie between 30 and 120 €/MWh, a few above 500. On a linear axis
the spikes squash everything else. A log axis shows multiplicative differences — but **a log axis cannot show zero or negative prices**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(train["price_eur_mwh"], bins=80); axes[0].set_title("linear")
axes[1].hist(np.log10(train["price_eur_mwh"].clip(lower=1)), bins=80); axes[1].set_title("log10 axis")
plt.show()

### ✏️ Your turn 3

Prepare the prices for a log axis.

Fill in the `TODO`s:
- `n_not_positive`: number of hours with a price of 0 or below, and `price_pos`: only the positive prices;
- `log_range`: `np.log10(price_pos.quantile(0.99) / price_pos.quantile(0.01))`, how many *orders of magnitude* the typical range (1 % to 99 % quantile) of the positive prices covers (2 decimals).

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
n_not_positive = np.nan
price_pos = price
log_range = np.nan
```
by
```python
n_not_positive = int((price <= 0).sum())
price_pos = price[price > 0]
log_range = np.log10(price_pos.quantile(0.99) / price_pos.quantile(0.01))
```

In [ ]:
price = train["price_eur_mwh"]

n_not_positive = np.nan       # TODO
price_pos = price              # TODO: only the positive prices
log_range = np.nan            # TODO

print("hours with price <= 0:", n_not_positive)
print("maximum price:", price.max())
print("orders of magnitude between q01 and q99:", round(log_range, 2))

### The daily rhythm that the daily mean hides

A line plot of **daily means** shows the seasons, but the *shape of the day* is lost.

In [ ]:
daily = train.set_index("timestamp")["load_mw"].resample("D").mean()
fig, ax = plt.subplots(figsize=(8, 2.8))
daily.plot(ax=ax); ax.set_ylabel("daily mean load (MW)")
plt.show()

### ✏️ Your turn 4

Compute the **average daily profile**: the mean `load_mw` for each hour of the day (0 – 23), using `groupby` on `train["timestamp"].dt.hour`.

Fill in the `TODO`s and report: the hour of the highest load (`peak_hour`), the hour of the lowest load (`trough_hour`),
and by how many per cent the peak exceeds the trough (`peak_over_trough_pct`, 1 decimal) = (peak / trough − 1) · 100.

**What to change** — edit the cell exactly like this, then run it:

Change `profile = pd.Series(dtype=float)` into `profile = train.groupby(train["timestamp"].dt.hour)["load_mw"].mean()`

In [ ]:
profile = pd.Series(dtype=float)     # TODO: mean load per hour of day
peak_hour = None
trough_hour = None
peak_over_trough_pct = np.nan

if len(profile):
    peak_hour, trough_hour = profile.idxmax(), profile.idxmin()
    peak_over_trough_pct = (profile.max() / profile.min() - 1) * 100
    print("peak hour:", peak_hour, "| mean load:", round(profile.max(), 1))
    print("trough hour:", trough_hour, "| mean load:", round(profile.min(), 1))
    print("peak exceeds trough by (%):", round(peak_over_trough_pct, 1))
    profile.plot(figsize=(7, 2.8), marker="o"); plt.ylabel("MW"); plt.xlabel("hour of day"); plt.show()

## 3. Boxplots

A boxplot shows five numbers per group: the median, the quartiles (box) and the whiskers (here: the furthest point within 1.5 × IQR
of the box). It shows *level* and *spread* together — but not how many observations each box is built on.

In [ ]:
train["month"] = train["timestamp"].dt.month_name()
fig, ax = plt.subplots(figsize=(9, 3.2))
train.boxplot(column="load_mw", by="month", ax=ax)          # months in the order pandas happens to produce
plt.suptitle(""); ax.set_xlabel(""); ax.set_ylabel("load (MW)")
plt.xticks(rotation=45); plt.show()

### ✏️ Your turn 5

Put the months in **calendar order** (an *ordinal* variable keeps its natural order; never alphabetical).

1. Fill in the `TODO`: `month_order` = the list of 12 month names from `"January"` to `"December"`.
   (Tip: `pd.date_range("2022-01-01", periods=12, freq="MS").month_name().tolist()`.)
2. The cell then computes per month: the number of observations `n`, the median, and the IQR of `load_mw`.
   Read off: the month with the **highest median**, the month with the **largest IQR**, and `n` for **June**.

**What to change** — edit the cell exactly like this, then run it:

Change `month_order = sorted(train["month"].unique())` into `month_order = pd.date_range("2022-01-01", periods=12, freq="MS").month_name().tolist()`

In [ ]:
month_order = sorted(train["month"].unique())      # TODO: calendar order, January ... December

groups = [train.loc[train["month"] == m, "load_mw"] for m in month_order if m in set(train["month"])]
stats = pd.DataFrame({
    "n": [len(g) for g in groups],
    "median": [g.median() for g in groups],
    "IQR": [g.quantile(0.75) - g.quantile(0.25) for g in groups],
}, index=[m for m in month_order if m in set(train["month"])]).round(1)

print(stats)
print("order of the boxes:", list(stats.index)[:4], "...")
print("highest median:", stats["median"].idxmax(), "| largest IQR:", stats["IQR"].idxmax())
print("observations in June:", stats.loc["June", "n"])

The boxes do not all rest on the same number of hours — in the training set, June–December appear once but January–May twice.
A boxplot hides this; **always report `n`** (or write it under each box).

### ✏️ Your turn 6

A *nominal* variable should be sorted by value instead. Sort the 24 hours of the day by the **median load at that hour**, from highest to lowest.

Fill in the `TODO`: `by_hour_median` = median `load_mw` per hour of day, sorted from highest to lowest. Report the three hours with the highest median and the median of the highest hour (nearest MW).

**What to change** — edit the cell exactly like this, then run it:

Change `by_hour_median = pd.Series(dtype=float)` into `by_hour_median = train.groupby(train["timestamp"].dt.hour)["load_mw"].median().sort_values(ascending=False)`

In [ ]:
by_hour_median = pd.Series(dtype=float)       # TODO: median per hour, sorted descending

if len(by_hour_median):
    print("three highest hours:", list(by_hour_median.index[:3]))
    print("median of the highest hour:", round(by_hour_median.iloc[0]))

## Summary

* KDE: bandwidth decides between noise and over-smoothing; it ignores boundaries and point masses (use a histogram there).
* Prices want a log axis — but zero and negative values cannot be drawn.
* A daily mean hides the daily rhythm; plot the profile.
* Boxplots: ordinal → natural order, nominal → sort by value, and always know how much data is behind each box.